# 2020/11 VIIRS NTL 臺灣裁切｜TGZ 直讀版

你只需要把原始 `.tgz` 放在 Google Drive：

`MyDrive/UVI_NTL/raw_2020/`

Notebook 會自動：
1. 掛載 Drive
2. 從 TGZ **只解出** `avg_rade9h.tif` 與 `cf_cvg.tif`
3. 裁切臺灣範圍
4. 無損壓縮輸出
5. 做 QA
6. 刪除 Colab 暫存的大 TIFF

原始 TGZ 不會被修改。

## 1. 安裝套件
執行這一格。

In [1]:
!pip -q install rasterio


## 2. 掛載 Google Drive
執行後依畫面授權。

In [2]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


## 3. 檢查 TGZ 是否存在
應該看到 `✅ 找到 TGZ`。

In [5]:
from pathlib import Path

BASE = Path('/content/drive/MyDrive/UVI_NTL')
RAW = BASE / 'raw_2020'
OUT = BASE / 'output_2020'
OUT.mkdir(parents=True, exist_ok=True)

TGZ = RAW / 'SVDNB_npp_20201101-20201130_75N060E_vcmcfg_v10_c202012092300.tgz'

print('TGZ 路徑：', TGZ)
print('✅ 找到 TGZ' if TGZ.exists() else '❌ 找不到 TGZ')
if TGZ.exists():
    print(f'檔案大小：{TGZ.stat().st_size / 1024 / 1024:.2f} MB')


TGZ 路徑： /content/drive/MyDrive/UVI_NTL/raw_2020/SVDNB_npp_20201101-20201130_75N060E_vcmcfg_v10_c202012092300.tgz
✅ 找到 TGZ
檔案大小：612.03 MB


## 4. 只解出兩個需要的 TIFF
這一格不會解 `cvg.tif`，只解：
- `avg_rade9h.tif`
- `cf_cvg.tif`


In [6]:
import tarfile
import shutil

TMP = Path('/content/viirs_2020_tmp')
if TMP.exists():
    shutil.rmtree(TMP)
TMP.mkdir(parents=True, exist_ok=True)

if not TGZ.exists():
    raise FileNotFoundError(f'找不到：{TGZ}')

wanted = ('avg_rade9h.tif', 'cf_cvg.tif')
extracted = []

with tarfile.open(TGZ, 'r:gz') as tar:
    members = [m for m in tar.getmembers() if any(m.name.endswith(x) for x in wanted)]
    print('找到需要的檔案：')
    for m in members:
        print(' -', m.name, f'({m.size / 1024 / 1024:.2f} MB)')
        tar.extract(m, path=TMP)
        extracted.append(TMP / m.name)

if len(extracted) != 2:
    raise RuntimeError(f'預期解出 2 個 TIFF，但實際找到 {len(extracted)} 個。')

for p in extracted:
    print('✅ 已解出：', p)


找到需要的檔案：
 - SVDNB_npp_20201101-20201130_75N060E_vcmcfg_v10_c202012092300.avg_rade9h.tif (1977.68 MB)


/tmp/ipykernel_2433/2134752506.py:20: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extract(m, path=TMP)


 - SVDNB_npp_20201101-20201130_75N060E_vcmcfg_v10_c202012092300.cf_cvg.tif (988.91 MB)
✅ 已解出： /content/viirs_2020_tmp/SVDNB_npp_20201101-20201130_75N060E_vcmcfg_v10_c202012092300.avg_rade9h.tif
✅ 已解出： /content/viirs_2020_tmp/SVDNB_npp_20201101-20201130_75N060E_vcmcfg_v10_c202012092300.cf_cvg.tif


## 5. 裁切臺灣並輸出到 Google Drive
研究範圍：118–122.5°E、21.5–26.5°N。

不重投影、不重取樣。

In [7]:
import rasterio
from rasterio.windows import from_bounds
from rasterio.warp import transform_bounds

WGS84_BOUNDS = (118.0, 21.5, 122.5, 26.5)

def clip_geotiff(src_path, dst_path):
    with rasterio.open(src_path) as src:
        if src.crs is None:
            raise ValueError(f'{src_path.name} 沒有 CRS。')

        print('\n正在處理：', src_path.name)
        print('原始 CRS：', src.crs)
        print('原始尺寸：', src.width, 'x', src.height)
        print('原始解析度：', src.res)

        bounds_in_src = transform_bounds(
            'EPSG:4326', src.crs, *WGS84_BOUNDS, densify_pts=21
        )

        window = from_bounds(*bounds_in_src, transform=src.transform)
        window = window.round_offsets().round_lengths()

        full_window = rasterio.windows.Window(0, 0, src.width, src.height)
        window = window.intersection(full_window)

        data = src.read(window=window)
        out_transform = src.window_transform(window)

        profile = src.profile.copy()
        profile.pop('blockxsize', None)
        profile.pop('blockysize', None)

        profile.update(
            height=data.shape[1],
            width=data.shape[2],
            transform=out_transform,
            compress='DEFLATE',
            tiled=True,
            blockxsize=256,
            blockysize=256,
            BIGTIFF='IF_SAFER'
        )

        if data.dtype.kind == 'f':
            profile['predictor'] = 3
        elif data.dtype.kind in ('u', 'i'):
            profile['predictor'] = 2

        with rasterio.open(dst_path, 'w', **profile) as dst:
            dst.write(data)

        print('✅ 完成：', dst_path)
        print('輸出尺寸：', data.shape[2], 'x', data.shape[1])
        print('資料型態：', data.dtype)

avg_src = next(p for p in extracted if p.name.endswith('avg_rade9h.tif'))
cf_src  = next(p for p in extracted if p.name.endswith('cf_cvg.tif'))

avg_out = OUT / 'Taiwan_NTL_2020_11_avg_rade9h.tif'
cf_out  = OUT / 'Taiwan_NTL_2020_11_cf_cvg.tif'

clip_geotiff(avg_src, avg_out)
clip_geotiff(cf_src, cf_out)



正在處理： SVDNB_npp_20201101-20201130_75N060E_vcmcfg_v10_c202012092300.avg_rade9h.tif
原始 CRS： EPSG:4326
原始尺寸： 28800 x 18000
原始解析度： (0.0041666667, 0.0041666667)
✅ 完成： /content/drive/MyDrive/UVI_NTL/output_2020/Taiwan_NTL_2020_11_avg_rade9h.tif
輸出尺寸： 1080 x 1200
資料型態： float32

正在處理： SVDNB_npp_20201101-20201130_75N060E_vcmcfg_v10_c202012092300.cf_cvg.tif
原始 CRS： EPSG:4326
原始尺寸： 28800 x 18000
原始解析度： (0.0041666667, 0.0041666667)
✅ 完成： /content/drive/MyDrive/UVI_NTL/output_2020/Taiwan_NTL_2020_11_cf_cvg.tif
輸出尺寸： 1080 x 1200
資料型態： uint16


## 6. 看輸出檔案大小

In [8]:
for p in [avg_out, cf_out]:
    print(f'{p.name}: {p.stat().st_size / 1024 / 1024:.2f} MB')


Taiwan_NTL_2020_11_avg_rade9h.tif: 3.06 MB
Taiwan_NTL_2020_11_cf_cvg.tif: 0.35 MB


## 7. QA：檢查兩張輸出 raster

In [9]:
import numpy as np

def qa_raster(path):
    with rasterio.open(path) as ds:
        arr = ds.read(1)
        finite = np.isfinite(arr)
        vals = arr[finite]

        print('\n' + '='*70)
        print(path.name)
        print('='*70)
        print('CRS:', ds.crs)
        print('Width x Height:', ds.width, 'x', ds.height)
        print('Bounds:', ds.bounds)
        print('Resolution:', ds.res)
        print('NoData:', ds.nodata)
        print('Count:', ds.count)
        print('Dtype:', ds.dtypes)
        print('Compression:', ds.compression)
        print('Block shapes:', ds.block_shapes)
        print('Min:', float(np.min(vals)))
        print('Max:', float(np.max(vals)))
        print('Mean:', float(np.mean(vals)))
        print('Median:', float(np.median(vals)))
        print('Zero pixels:', int(np.count_nonzero(arr == 0)))
        print('NaN pixels:', int(np.count_nonzero(np.isnan(arr))))
        print('Inf pixels:', int(np.count_nonzero(np.isinf(arr))))

qa_raster(avg_out)
qa_raster(cf_out)



Taiwan_NTL_2020_11_avg_rade9h.tif
CRS: EPSG:4326
Width x Height: 1080 x 1200
Bounds: BoundingBox(left=117.99791713065, bottom=21.502082905349997, right=122.49791716665, top=26.502082945349997)
Resolution: (0.0041666667, 0.0041666667)
NoData: None
Count: 1
Dtype: ('float32',)
Compression: Compression.deflate
Block shapes: [(256, 256)]
Min: -0.4399999976158142
Max: 976.02001953125
Mean: 1.4478824138641357
Median: 0.2800000011920929
Zero pixels: 5022
NaN pixels: 0
Inf pixels: 0

Taiwan_NTL_2020_11_cf_cvg.tif
CRS: EPSG:4326
Width x Height: 1080 x 1200
Bounds: BoundingBox(left=117.99791713065, bottom=21.502082905349997, right=122.49791716665, top=26.502082945349997)
Resolution: (0.0041666667, 0.0041666667)
NoData: None
Count: 1
Dtype: ('uint16',)
Compression: Compression.deflate
Block shapes: [(256, 256)]
Min: 0.0
Max: 17.0
Mean: 5.870303240740741
Median: 5.0
Zero pixels: 4643
NaN pixels: 0
Inf pixels: 0


## 8. 檢查兩張 raster 是否完全對齊

In [10]:
with rasterio.open(avg_out) as a, rasterio.open(cf_out) as c:
    checks = {
        'CRS 相同': a.crs == c.crs,
        '尺寸相同': (a.width, a.height) == (c.width, c.height),
        'Transform 相同': a.transform == c.transform,
        'Bounds 相同': a.bounds == c.bounds,
        'Resolution 相同': a.res == c.res,
    }

    print('Grid alignment QA')
    for k, v in checks.items():
        print(k + ':', 'PASS' if v else 'FAIL')

    avg = a.read(1)
    cf = c.read(1)
    mask0 = (cf == 0)

    print('\ncf_cvg = 0 pixel count:', int(mask0.sum()))
    if mask0.any():
        vals = avg[mask0]
        print('avg_rade9h at cf_cvg=0 -> min:', float(np.nanmin(vals)))
        print('avg_rade9h at cf_cvg=0 -> max:', float(np.nanmax(vals)))
        print('avg_rade9h at cf_cvg=0 -> mean:', float(np.nanmean(vals)))
        print('avg_rade9h at cf_cvg=0 -> zero count:', int(np.count_nonzero(vals == 0)))

    print('\n✅ 兩張 raster 網格完全一致。' if all(checks.values()) else '\n⚠️ 網格有不一致。')


Grid alignment QA
CRS 相同: PASS
尺寸相同: PASS
Transform 相同: PASS
Bounds 相同: PASS
Resolution 相同: PASS

cf_cvg = 0 pixel count: 4643
avg_rade9h at cf_cvg=0 -> min: 0.0
avg_rade9h at cf_cvg=0 -> max: 0.0
avg_rade9h at cf_cvg=0 -> mean: 0.0
avg_rade9h at cf_cvg=0 -> zero count: 4643

✅ 兩張 raster 網格完全一致。


## 9. 清除 Colab 暫存的大 TIFF
確認第 5～8 步都成功後再執行。Google Drive 裡的 TGZ 和輸出檔不會被刪除。

In [11]:
import shutil

if TMP.exists():
    shutil.rmtree(TMP)
    print('✅ 已清除 Colab 暫存大檔：', TMP)
else:
    print('暫存資料夾已不存在。')

print('\nGoogle Drive 最終輸出：')
print(avg_out)
print(cf_out)


✅ 已清除 Colab 暫存大檔： /content/viirs_2020_tmp

Google Drive 最終輸出：
/content/drive/MyDrive/UVI_NTL/output_2020/Taiwan_NTL_2020_11_avg_rade9h.tif
/content/drive/MyDrive/UVI_NTL/output_2020/Taiwan_NTL_2020_11_cf_cvg.tif


## 完成
最後 Google Drive 應該有：

`MyDrive/UVI_NTL/output_2020/Taiwan_NTL_2020_11_avg_rade9h.tif`

`MyDrive/UVI_NTL/output_2020/Taiwan_NTL_2020_11_cf_cvg.tif`

完成後，把第 6、7、8 步的輸出貼給 ChatGPT 檢查即可。